# ModelScope

> **Run this notebook on Colab T4 GPU** — Runtime → Change runtime type → T4 GPU.

Benchmarks **Llama-3.2-3B** and **Gemma-3-4b** across 8 quantisation variants  
(fp16 / int8 / int4 / int4-nf4). Outputs latency/throughput CSVs, quality CSVs,  
three plots, and `results/recommendation.json`.

Mount Google Drive and clone the repo from GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ← Update to your GitHub username/repo before running
REPO = "YOUR_GITHUB_USERNAME/modelscope"
!git clone https://github.com/{REPO}.git /content/modelscope
%cd /content/modelscope

Install all Python dependencies from `requirements.txt`.

In [ ]:
!pip install -q -r requirements.txt

Authenticate with HuggingFace using the `HF_TOKEN` Colab secret (Secrets panel → left sidebar).

In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

Confirm the T4 GPU is visible and CUDA is working before running any models.

In [ ]:
!nvidia-smi

import torch
print(f"\nCUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Device         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {props.total_memory / 1024**3:.1f} GB")

Run the full latency and throughput benchmark for all 8 variants; writes `results/benchmark_results.csv`.

In [ ]:
from benchmarks.runner import run_all_benchmarks
run_all_benchmarks()

Run MMLU accuracy, output consistency, and semantic relevance evaluation; writes `results/quality_results.csv`.

In [ ]:
from eval.harness import run_full_eval
run_full_eval()

Compute the Pareto frontier, generate all three result plots, and write `results/recommendation.json`.

In [ ]:
from analysis.pareto import load_results, find_pareto_frontier
from analysis.visualize import (
    plot_memory_vs_throughput,
    plot_quality_vs_speed,
    plot_pareto_frontier,
)
from analysis.recommend import generate_recommendation

df = load_results()
df = find_pareto_frontier(df)

plot_memory_vs_throughput(df)
plot_quality_vs_speed(df)
plot_pareto_frontier(df)

generate_recommendation(df)
print("Done — plots in results/plots/, recommendation in results/recommendation.json")

Display the three saved result plots inline.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

plots = [
    ("Memory vs Throughput", "results/plots/memory_vs_throughput.png"),
    ("Quality vs Speed",     "results/plots/quality_vs_speed.png"),
    ("Pareto Frontier",      "results/plots/pareto_frontier.png"),
]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (title, path) in zip(axes, plots):
    ax.imshow(mpimg.imread(path))
    ax.set_title(title, fontsize=12)
    ax.axis("off")
plt.tight_layout()
plt.show()

Copy the full `results/` directory to Google Drive under a timestamped folder for persistence.

In [ ]:
import shutil
from pathlib import Path
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
dest = Path(f"/content/drive/MyDrive/modelscope_results_{timestamp}")
dest.mkdir(parents=True, exist_ok=True)

shutil.copytree("results", dest / "results", dirs_exist_ok=True)
print(f"Results saved to {dest}")

Print the final deployment recommendation for each priority category.

In [ ]:
import json
from pathlib import Path

rec = json.loads(Path("results/recommendation.json").read_text())
for category, info in rec.items():
    print(f"\n{'=' * 52}")
    print(f"  {category.upper().replace('_', ' ')}")
    print(f"{'=' * 52}")
    for key in ("variant", "model", "config", "memory_mb",
                "tokens_per_sec_batch1", "mmlu_accuracy", "rationale"):
        print(f"  {key:<22}: {info[key]}")